# Smart Lost & Found - train on a COMBINED dataset

Run this on **Google Colab** with a GPU (`Runtime > Change runtime type > T4 GPU`).

It combines the datasets listed in `dataset_sources.yaml` (the original Lost and Found dataset plus earbuds, student ID, wallet, glasses and desk-object datasets) into one training set with `scripts/merge_datasets.py`, trains YOLO11 on it, evaluates it and downloads the new `best.pt`.

Training takes roughly 1-2 hours on a T4. If Colab disconnects, re-run from step 5 - the merged dataset is rebuilt in a few minutes.

## 1. Install packages and check the GPU

In [ ]:
%pip install -q ultralytics roboflow

import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU - enable one under Runtime > Change runtime type, then restart.')

## 2. Get the project code

**Option A:** clone the GitHub repository (works if it is public, or after signing in).

**Option B:** zip the `lost_and_found` folder on your computer, upload the zip with the folder icon on the left, then run the second cell instead.

In [ ]:
import os
from pathlib import Path

REPO = 'https://github.com/Kebinging/lost_and_found.git'
if not Path('/content/lost_and_found').exists():
    !git clone -q {REPO} /content/lost_and_found
os.chdir('/content/lost_and_found')
print('Working in', os.getcwd())

In [ ]:
# Option B only: unzip an uploaded copy of the project instead of cloning.
# import zipfile, os
# zipfile.ZipFile('/content/lost_and_found.zip').extractall('/content')
# os.chdir('/content/lost_and_found')

## 3. Roboflow API key

Add your key as a Colab secret named `ROBOFLOW_API_KEY` (key icon in the left sidebar) and allow this notebook to use it. The key is never written into the notebook.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['ROBOFLOW_API_KEY'] = userdata.get('ROBOFLOW_API_KEY')
    print('API key loaded from Colab secrets.')
except Exception as error:
    print('Could not read the secret:', error)

## 4. (Optional) Your own campus photos

Labelled photos of real items are the best way to make the model work on your campus. Label them in Roboflow (export **YOLOv8**) or Label Studio, upload the folder as `datasets/sources/campus`, and un-comment the `campus` source in `dataset_sources.yaml`. They become a separate **real-world test set** that is never trained on.

## 5. Download and merge the datasets

`--pseudo-label` uses the current `models/best.pt` + the COCO model to label objects a dataset does not label itself (for example a phone lying next to a wallet), so they are not learned as background. Add `--skip glasses` to leave a source out.

In [ ]:
!python scripts/merge_datasets.py --download --pseudo-label

In [ ]:
# Look at a few merged training images with their labels before spending an hour training.
import random, yaml
from PIL import Image, ImageDraw
from IPython.display import display

DATA = Path('datasets/combined')
names = yaml.safe_load((DATA / 'data.yaml').read_text())['names']
for image_path in random.Random(0).sample(sorted((DATA / 'train' / 'images').iterdir()), 4):
    image = Image.open(image_path).convert('RGB')
    draw, (w, h) = ImageDraw.Draw(image), image.size
    for line in (DATA / 'train' / 'labels' / f'{image_path.stem}.txt').read_text().splitlines():
        c, x, y, bw, bh = line.split(); x, y, bw, bh = (float(v) for v in (x, y, bw, bh))
        draw.rectangle(((x - bw / 2) * w, (y - bh / 2) * h, (x + bw / 2) * w, (y + bh / 2) * h), outline='red', width=4)
        draw.text(((x - bw / 2) * w + 4, (y - bh / 2) * h + 4), names[int(c)], fill='red')
    image.thumbnail((480, 480)); print(image_path.name); display(image)

## 6. Train

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')   # small and fast enough for a laptop CPU at prediction time
results = model.train(data=str(DATA / 'data.yaml'), epochs=80, imgsz=640, batch=16,
                      patience=15, seed=42)
RUN_DIR = Path(results.save_dir)
print('Run folder:', RUN_DIR)

## 7. Evaluate

The **test** split was never used for training. If you added campus photos, the **real-world test** shows how the model does on your own photos - quote both in the report, and expect the real-world numbers to be lower.

In [ ]:
best = YOLO(str(RUN_DIR / 'weights' / 'best.pt'))

def report(data_yaml, title):
    metrics = best.val(data=str(data_yaml), split='test', plots=True)
    print(f'
{title}')
    print(f"{'class':<12}{'precision':>10}{'recall':>10}{'mAP50':>10}{'mAP50-95':>10}")
    print(f"{'all':<12}{metrics.box.mp:>10.3f}{metrics.box.mr:>10.3f}{metrics.box.map50:>10.3f}{metrics.box.map:>10.3f}")
    for i, c in enumerate(metrics.box.ap_class_index):
        p, r, ap50, ap = metrics.box.class_result(i)
        print(f'{best.names[int(c)]:<12}{p:>10.3f}{r:>10.3f}{ap50:>10.3f}{ap:>10.3f}')
    return Path(metrics.save_dir)

EVAL_DIR = report(DATA / 'data.yaml', 'Test split')
if (DATA / 'data_real_test.yaml').exists():
    report(DATA / 'data_real_test.yaml', 'Real-world test (own campus photos)')

In [ ]:
# Confusion matrix, PR curve and sample predictions (screenshot these for the report).
from IPython.display import Image as ShowImage
for pattern in ['confusion_matrix.png', '*PR_curve.png', 'val_batch0_pred.jpg']:
    for plot in sorted(EVAL_DIR.glob(pattern)):
        print(plot.name); display(ShowImage(filename=str(plot), width=700))
print((DATA / 'SOURCES.md').read_text())   # dataset references for the report

## 8. Download `best.pt`

Save it as `models/best.pt` in the project (keep the old one as `models/best_v1.pt` so you can compare them). The app reads the class names from the model, so the new categories are detected by YOLO automatically - CLIP then only has to step in for items outside them.

In [ ]:
from google.colab import files
files.download(str(RUN_DIR / 'weights' / 'best.pt'))